# Bag of Words Models

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
import matplotlib.pyplot as plt

corpus = [
    'the quick brown fox',
    'the fast brown dog',
    'the quick red fox',
    "the brown happy dog jumps over the lazy fox"
]

count_vectorizer = CountVectorizer()

term_matrix = count_vectorizer.fit_transform(corpus)
term_matrix_dense = term_matrix.toarray()

feature_names = count_vectorizer.get_feature_names_out()

toy_example = pd.DataFrame(term_matrix_dense, columns=feature_names, index=[f'Doc {i+1}' for i in range(len(corpus))])
toy_example

In [ ]:
# Read in the data

df = pd.read_csv("https://github.com/febse/data/raw/refs/heads/main/ta/reviews.csv")
df.head()


In [ ]:
# Remove missing values
df.dropna(inplace=True)

# Drop reviews with neutral ratings
df = df[df['Rating'] != 3]

df["positive"] = np.where(df['Rating'] > 3, 1, 0)
df.Reviews.head(10)

In [ ]:
# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(df['Reviews'],
                                                    df['positive'],
                                                    random_state=0)

## Logistic Regression Classifier

In [ ]:
# Create bag-of-words features using CountVectorizer

vectorizer = CountVectorizer(
    min_df=5
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f'Number of features: {X_train_vec.shape[1]}')
print(f'Training set shape: {X_train_vec.shape}')
print(f'Test set shape: {X_test_vec.shape}')

In [ ]:
# Train logistic regression classifier
logreg = LogisticRegression(max_iter=1000, random_state=0)
logreg.fit(X_train_vec, y_train)

# Make predictions
y_pred = logreg.predict(X_test_vec)
y_pred_proba = logreg.predict_proba(X_test_vec)[:, 1]

# Calculate accuracy and AUC
accuracy = logreg.score(X_test_vec, y_test)
auc_score = roc_auc_score(y_test, y_pred_proba)

print(f'Accuracy: {accuracy:.4f}')
print(f'AUC Score: {auc_score:.4f}')

### Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Create and display confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Negative', 'Positive'])

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
plt.title('Confusion Matrix for Sentiment Classification')
plt.show()

print(f'\nConfusion Matrix:')
print(f'True Negatives: {cm[0,0]}')
print(f'False Positives: {cm[0,1]}')
print(f'False Negatives: {cm[1,0]}')
print(f'True Positives: {cm[1,1]}')

### ROC Curve

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)

# Plot ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, linewidth=2, label=f'ROC curve (AUC = {auc_score:.4f})')
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random classifier')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve for Sentiment Classification', fontsize=14)
plt.legend(loc='lower right', fontsize=11)
plt.grid(alpha=0.3)
plt.show()

### Most Important Features

In [ ]:
# Get feature names and coefficients
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = logreg.coef_[0]

# Create a dataframe with features and their coefficients
feature_coef = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefficients
}).sort_values('coefficient', ascending=False)

# Top 20 features associated with positive sentiment
print("Top 20 features most associated with POSITIVE sentiment:")
print(feature_coef.head(20).to_string(index=False))

print("\n" + "="*60 + "\n")

# Top 20 features associated with negative sentiment
print("Top 20 features most associated with NEGATIVE sentiment:")
print(feature_coef.tail(20).to_string(index=False))

In [ ]:
# Visualize top features for each class
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Top positive features
top_positive = feature_coef.head(15)

ax1.barh(range(len(top_positive)), top_positive['coefficient'], color='green', alpha=0.7)
ax1.set_yticks(range(len(top_positive)))
ax1.set_yticklabels(top_positive['feature'])
ax1.set_xlabel('Coefficient', fontsize=12)
ax1.set_title('Top 15 Features for Positive Sentiment', fontsize=14)
ax1.invert_yaxis()

# Top negative features
top_negative = feature_coef.tail(15).sort_values('coefficient', ascending=True)

ax2.barh(range(len(top_negative)), top_negative['coefficient'], color='red', alpha=0.7)
ax2.set_yticks(range(len(top_negative)))
ax2.set_yticklabels(top_negative['feature'])
ax2.set_xlabel('Coefficient', fontsize=12)
ax2.set_title('Top 15 Features for Negative Sentiment', fontsize=14)
ax2.invert_yaxis()

plt.tight_layout()
plt.show()